# Evaluating Third-Party Agent Trajectories with NeMo Agent Toolkit

This notebook demonstrates **ATIF as an interoperability contract** between
third-party agent frameworks and the NeMo Agent Toolkit evaluation harness.

We generate real ATIF trajectories using
[Harbor](https://github.com/harbor-framework/harbor) running the
`mini-swe-agent` on [BFCL](https://gorilla.cs.berkeley.edu/leaderboard.html)
(Berkeley Function-Calling Leaderboard) tasks, then evaluate them
through `nvidia-nat-eval` using both custom and RAGAS evaluators.

```
Harbor CLI  -->  ATIF JSON  -->  NAT ATIF Models  -->  EvaluationHarness  -->  Results
 (any agent)     (standard)      (Pydantic parse)      (custom + RAGAS)
```

**What this proves:**
- Any agent framework that outputs ATIF trajectories can be evaluated by NeMo Agent Toolkit
- Both lightweight custom evaluators and LLM-as-judge evaluators work on the same ATIF data
- No NeMo Agent Toolkit workflow is needed — `nvidia-nat-eval` is a standalone component

**Prerequisites:**
- **Docker** — Harbor runs agents inside containers
- **`NVIDIA_API_KEY`** environment variable — used by the NIM model endpoint and RAGAS evaluators
- `nvidia-nat-eval`, `nvidia-nat-ragas`, `nvidia-nat-langchain` — installed in Section 1

## 1. Install Dependencies

In [ ]:
# Install nat-eval, RAGAS evaluator plugin, and langchain (needed by RAGAS for LLM wrapping).
#
# For development on this branch, install from local source:
!uv pip install -q \
    -e ../../packages/nvidia_nat_eval \
    -e ../../packages/nvidia_nat_ragas \
    -e ../../packages/nvidia_nat_langchain
#
# For released versions, use:
# !uv pip install nvidia-nat-eval nvidia-nat-ragas nvidia-nat-langchain

## 2. Generate ATIF Trajectories with Harbor

[Harbor](https://github.com/harbor-framework/harbor) runs agents against
benchmark datasets inside Docker containers and outputs ATIF-formatted
trajectory logs.

Configure the run parameters below, then execute the next cell to install
Harbor and run the agent. The first run may take several minutes while Docker
images are pulled and built.

In [ ]:
import os
import subprocess

HARBOR_DATASET = "bfcl"
HARBOR_AGENT = "mini-swe-agent"
HARBOR_MODEL = "nvidia_nim/meta/llama-3.3-70b-instruct"
HARBOR_N_TASKS = 5

assert "NVIDIA_API_KEY" in os.environ, (
    "NVIDIA_API_KEY must be set. "
    "Export it before starting Jupyter: export NVIDIA_API_KEY='nvapi-...'"
)

In [ ]:
import json
from pathlib import Path

subprocess.run(
    ["uv", "tool", "install", "harbor"],
    check=True,
    capture_output=True,
)
print("Harbor installed.\n")

env = {
    **os.environ,
    "NVIDIA_NIM_API_KEY": os.environ["NVIDIA_API_KEY"],
    "MSWEA_COST_TRACKING": "ignore_errors",
}
print(
    f"Running: harbor run --dataset {HARBOR_DATASET} "
    f"--agent {HARBOR_AGENT} --n-tasks {HARBOR_N_TASKS}\n"
)
result = subprocess.run(
    [
        "harbor", "run",
        "--dataset", HARBOR_DATASET,
        "--agent", HARBOR_AGENT,
        "--model", HARBOR_MODEL,
        "--n-tasks", str(HARBOR_N_TASKS),
    ],
    env=env,
    capture_output=True,
    text=True,
    check=False,
)
stdout = result.stdout[-1000:] if len(result.stdout) > 1000 else result.stdout
print(stdout)
if result.returncode != 0:
    print(f"Harbor exited with code {result.returncode}")
    if result.stderr:
        print(result.stderr[-500:])

## 3. Load ATIF Trajectories from Harbor Output

Harbor writes one `trajectory.json` per trial under
`jobs/<timestamp>/<task-name>/agent/`. Each file is a valid ATIF document.
We scan the latest job directory and load all trajectories that contain
at least one agent step.

In [ ]:
from nat.data_models.atif import ATIFTrajectory

jobs_dir = Path("jobs")
latest_job = max(jobs_dir.iterdir(), key=lambda p: p.stat().st_mtime)

trajectories: dict[str, ATIFTrajectory] = {}
for trial_dir in sorted(latest_job.iterdir()):
    traj_file = trial_dir / "agent" / "trajectory.json"
    if not traj_file.exists():
        continue
    traj_data = json.loads(traj_file.read_text())
    steps = traj_data.get("steps", [])
    agent_steps = [s for s in steps if s.get("source") == "agent"]
    if not agent_steps:
        continue
    name = trial_dir.name.rsplit("__", 1)[0]
    trajectories[name] = ATIFTrajectory.model_validate(traj_data)

print(f"Loaded {len(trajectories)} trajectories from {latest_job}\n")
for name, traj in trajectories.items():
    agent_steps = [s for s in traj.steps if s.source == "agent"]
    tool_count = sum(len(s.tool_calls or []) for s in agent_steps)
    print(
        f"  {name}: {len(traj.steps)} steps, "
        f"{len(agent_steps)} agent turns, "
        f"{tool_count} tool calls, "
        f"{traj.final_metrics.total_prompt_tokens}/"
        f"{traj.final_metrics.total_completion_tokens} tokens"
    )

assert trajectories, (
    "No trajectories with agent steps found. "
    "Check the Harbor output above for errors."
)

## 4. Build `AtifEvalSample` Objects

Each sample pairs a trajectory with:
- **`output_obj`**: what the agent actually produced (extracted from the bash
  command that writes `result.json`)
- **`expected_output_obj`**: set to the agent output (ground truth is not
  available without the Harbor verifier result files)

For BFCL tasks, the agent writes a JSON array to `/app/result.json` via a bash
command. We extract the agent's answer from the tool call arguments.

In [ ]:
import re


def extract_bfcl_output(trajectory: ATIFTrajectory) -> str:
    """Extract the JSON written to /app/result.json from agent tool calls."""
    for step in trajectory.steps:
        if step.source != "agent" or not step.tool_calls:
            continue
        for tc in step.tool_calls:
            cmd = tc.arguments.get("command", "")
            if "/app/result.json" in cmd and "COMPLETE_TASK" not in cmd:
                match = re.search(
                    r"echo\s+'?(.*?)'?\s*>\s*/app/result\.json", cmd
                )
                if match:
                    return match.group(1)
    return ""


for name, traj in trajectories.items():
    output = extract_bfcl_output(traj)
    print(f"  {name}: {output[:100]}")

In [ ]:
from nat.plugins.eval.evaluator.atif_evaluator import AtifEvalSample

atif_samples = []
for name, traj in trajectories.items():
    agent_output = extract_bfcl_output(traj)
    sample = AtifEvalSample(
        item_id=name,
        trajectory=traj,
        expected_output_obj=agent_output,
        output_obj=agent_output,
    )
    atif_samples.append(sample)
    print(f"  {name}: output={agent_output[:80]}")

print(f"\nCreated {len(atif_samples)} AtifEvalSample(s)")

## 5. Custom Evaluators (No API Keys Required)

We define two lightweight evaluators that run locally without any LLM calls.
These inherit from `AtifBaseEvaluator` and implement a single method:
`async def evaluate_atif_item(self, sample) -> EvalOutputItem`.

- **BFCLFunctionCallEvaluator**: Parses the JSON output and checks whether the
  agent called the correct function(s) with the right arguments.
- **TrajectoryEfficiencyEvaluator**: Scores based on how concisely the agent
  solved the task (fewer steps and tokens = higher score).

In [ ]:
from nat.data_models.evaluator import EvalOutputItem
from nat.plugins.eval.evaluator.atif_base_evaluator import AtifBaseEvaluator
from nat.plugins.eval.evaluator.atif_evaluator import AtifEvalSample


class BFCLFunctionCallEvaluator(AtifBaseEvaluator):
    """Evaluates whether the agent called the correct function(s) with correct arguments.

    Scoring:
    - 1.0: function names AND arguments match expected output exactly
    - 0.5: function names match but arguments differ
    - 0.0: function names don't match or output is unparseable
    """

    def __init__(self, max_concurrency: int = 4) -> None:
        super().__init__(max_concurrency=max_concurrency)

    def _parse_bfcl_json(self, raw: str) -> list[dict] | None:
        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                return parsed
        except (json.JSONDecodeError, TypeError):
            pass
        return None

    def _extract_function_names(self, calls: list[dict]) -> set[str]:
        return {name for call in calls for name in call.keys()}

    async def evaluate_atif_item(self, sample: AtifEvalSample) -> EvalOutputItem:
        """Score one ATIF sample on function-call correctness."""
        expected = self._parse_bfcl_json(str(sample.expected_output_obj))
        generated = self._parse_bfcl_json(str(sample.output_obj))

        if expected is None or generated is None:
            return EvalOutputItem(
                id=sample.item_id,
                score=0.0,
                reasoning={
                    "error": "Could not parse JSON",
                    "raw_output": str(sample.output_obj)[:200],
                },
            )

        if expected == generated:
            score = 1.0
            detail = "exact_match"
        elif self._extract_function_names(expected) == self._extract_function_names(generated):
            score = 0.5
            detail = "function_names_match_args_differ"
        else:
            score = 0.0
            detail = "function_names_mismatch"

        return EvalOutputItem(
            id=sample.item_id,
            score=score,
            reasoning={
                "detail": detail,
                "expected_functions": sorted(self._extract_function_names(expected)) if expected else [],
                "generated_functions": sorted(self._extract_function_names(generated)) if generated else [],
                "expected": expected,
                "generated": generated,
            },
        )


class TrajectoryEfficiencyEvaluator(AtifBaseEvaluator):
    """Scores trajectory efficiency: fewer agent steps and tokens = higher score.

    Normalizes to [0, 1] using configurable upper bounds.
    """

    def __init__(
        self, max_steps: int = 10, max_tokens: int = 20000, max_concurrency: int = 4
    ) -> None:
        super().__init__(max_concurrency=max_concurrency)
        self._max_steps = max_steps
        self._max_tokens = max_tokens

    async def evaluate_atif_item(self, sample: AtifEvalSample) -> EvalOutputItem:
        """Score one ATIF sample on trajectory efficiency."""
        traj = sample.trajectory
        agent_steps = [s for s in traj.steps if s.source == "agent"]
        total_tokens = (
            (traj.final_metrics.total_prompt_tokens or 0)
            + (traj.final_metrics.total_completion_tokens or 0)
        ) if traj.final_metrics else 0

        step_score = max(0.0, 1.0 - len(agent_steps) / self._max_steps)
        token_score = max(0.0, 1.0 - total_tokens / self._max_tokens)
        score = round((step_score + token_score) / 2, 3)

        return EvalOutputItem(
            id=sample.item_id,
            score=score,
            reasoning={
                "agent_steps": len(agent_steps),
                "total_tokens": total_tokens,
                "step_score": round(step_score, 3),
                "token_score": round(token_score, 3),
            },
        )


print("Custom evaluators defined: BFCLFunctionCallEvaluator, TrajectoryEfficiencyEvaluator")

In [ ]:
from nat.plugins.eval.runtime.eval_harness import EvaluationHarness

harness = EvaluationHarness()

custom_results = await harness.evaluate(
    evaluators={
        "bfcl_function_call": BFCLFunctionCallEvaluator(),
        "trajectory_efficiency": TrajectoryEfficiencyEvaluator(),
    },
    atif_samples=atif_samples,
)

print("=" * 70)
print("Custom Evaluator Results")
print("=" * 70)
for eval_name, eval_output in custom_results.items():
    print(f"\n--- {eval_name} (avg={eval_output.average_score:.3f}) ---")
    for item in eval_output.eval_output_items:
        print(f"  {item.id}: score={item.score}")
        print(f"    {item.reasoning}")

## 6. RAGAS Evaluators (Requires `NVIDIA_API_KEY`)

For LLM-as-judge evaluation, we use the RAGAS `AnswerAccuracy` metric via the
`nvidia-nat-ragas` plugin. This requires an NVIDIA API key for NIM-hosted LLM
inference.

The same ATIF trajectories and `AtifEvalSample` objects are reused — only the
evaluator changes.

In [ ]:
from nat.data_models.config import Config
from nat.runtime.loader import PluginTypes
from nat.runtime.loader import discover_and_register_plugins
from nat.utils.data_models.schema_validator import validate_schema

discover_and_register_plugins(PluginTypes.CONFIG_OBJECT)

config_dict = {
    "llms": {
        "eval_llm": {
            "_type": "nim",
            "model_name": "nvidia/llama-3.3-nemotron-super-49b-v1",
            "temperature": 0.0,
        },
    },
    "eval": {
        "general": {"max_concurrency": 1},
        "evaluators": {
            "accuracy": {
                "_type": "ragas",
                "llm_name": "eval_llm",
                "metric": "AnswerAccuracy",
                "enable_atif_evaluator": True,
            },
        },
    },
}

config = validate_schema(config_dict, Config)
print(f"Evaluators configured: {list(config.eval.evaluators.keys())}")

In [ ]:
from nat.plugins.eval.evaluator.atif_evaluator import AtifEvaluator
from nat.plugins.eval.runtime.builder import WorkflowEvalBuilder

eval_builder = WorkflowEvalBuilder(
    general_config=config.general,
    eval_general_config=config.eval.general,
)
await eval_builder.__aenter__()
await eval_builder.populate_builder(config, skip_workflow=True)

atif_evaluators = {}
for name in config.eval.evaluators:
    evaluator_info = eval_builder.get_evaluator(name)
    if isinstance(evaluator_info, AtifEvaluator):
        atif_evaluators[name] = evaluator_info
        print(f"  [ATIF] {name}: {evaluator_info.description}")

ragas_results = await harness.evaluate(
    evaluators=atif_evaluators,
    atif_samples=atif_samples,
)

await eval_builder.__aexit__(None, None, None)

print("\n" + "=" * 70)
print("RAGAS Evaluation Results")
print("=" * 70)
for eval_name, eval_output in ragas_results.items():
    print(f"\n--- {eval_name} (avg={eval_output.average_score:.3f}) ---")
    for item in eval_output.eval_output_items:
        print(f"  {item.id}: score={item.score}")
        if item.reasoning:
            reasoning_str = str(item.reasoning)
            print(
                f"    reasoning: {reasoning_str[:200]}"
                f"{'...' if len(reasoning_str) > 200 else ''}"
            )

## 7. Summary

This notebook demonstrated end-to-end interoperability between a third-party
agent framework (Harbor) and the NeMo Agent Toolkit evaluation system:

1. **Harbor generated ATIF trajectories** — `mini-swe-agent` ran BFCL
   function-calling tasks and produced standard ATIF JSON output.

2. **NeMo Agent Toolkit parsed and validated** — `ATIFTrajectory.model_validate()`
   loaded the Harbor output into typed Pydantic models.

3. **Custom evaluators scored without LLMs** — `BFCLFunctionCallEvaluator`
   checked function-call correctness; `TrajectoryEfficiencyEvaluator` measured
   step and token efficiency. No API keys needed.

4. **RAGAS evaluators scored with LLM-as-judge** — The same `AtifEvalSample`
   objects were evaluated by RAGAS `AnswerAccuracy` using NIM endpoints.

| Component | What We Used |
|---|---|
| **Agent framework** | Harbor (`mini-swe-agent`) |
| **Benchmark** | BFCL (Berkeley Function-Calling Leaderboard) |
| **Model** | `meta/llama-3.3-70b-instruct` (via NVIDIA NIM) |
| **Interchange format** | ATIF v1.2 |
| **Eval harness** | `nvidia-nat-eval` (`EvaluationHarness`) |
| **Custom evaluators** | `BFCLFunctionCallEvaluator`, `TrajectoryEfficiencyEvaluator` |
| **LLM evaluators** | RAGAS `AnswerAccuracy` via `nvidia-nat-ragas` |

Any agent framework that outputs ATIF can be evaluated this way — the eval
harness is agnostic to how the trajectory was produced.